# How well is EI optimized? A deterministic, gradient-refined EI as the base policy

## Description

<!-- Written 2026-11-12 before the first run (pre-registration). Issue #13; research_log/decisions/0007; Tim's question on candidate selection. -->

**Question.**
- π₀ (EI) so far is the argmax of the PFN's EI over a fresh random candidate set at every (task, step, replicate). The set is 1024 shifted Sobol points plus 64 Gaussian perturbations of the incumbent at each of σ ∈ {0.01, 0.05, 0.2}, with no refinement.
- That makes π₀ stochastic. On noise-free tasks, this is the *only* noise in the round-1 labels: the continuations differ only in EI's candidate sets.
- It also confounds the advantage over EI with EI's own optimizer error. A candidate with higher EI than the sampled argmax "beats EI", although EI properly optimized would have chosen it.
- A scratch check (600 labeled states, not tracked) gave the motivation:
  - the sampled argmax reaches about 99.5% of the best EI found (median), so the value lost is small;
  - but picks from two replicate candidate sets differ by a median distance of 0.005–0.018. That is large where the incumbent is within one PFN bin of f\* (57% of the labeled states; 0007 §3).
- This experiment asks:
  1. how suboptimal and how stochastic the current EI is;
  2. whether a deterministic, gradient-refined EI loses anything in BO;
  3. whether it can be π₀ for relabeling. With deterministic π₀ on noise-free tasks, a label needs a single continuation and no cross-fitting.

**Setup** (`experimental/ei_opt.py`, `tasks/ei_opt.py`). Four variants of EI, the same PFN (pfn_large, fp32), the same EI closed form:
- `sampled`: the current π₀.
- `fixed`: candidate sets of the same sizes, but a function of the data only. The Sobol set is unshifted, and the perturbation directions around the incumbent are fixed: one draw shared by all lanes and steps.
- `sampled+refine`, `fixed+refine`:
  - the top 8 candidates by EI are the starts;
  - Adam on x through the PFN (it is differentiable in x) maximizes log EI: 100 steps, lr 0.01 cosine-annealed to 1e-4, clamped to [0, 1];
  - the best iterate per start is kept (iterates at an evaluated point are excluded);
  - the pick is the best of the refined starts and the base argmax, so refinement never lowers EI.
- *Part A (states).*
  - On the first 2000 labeled states of the pinned round-1 label set (split 4, d = 2), every variant picks under 4 replicate candidate sets.
  - Per pick: its EI relative to the best EI any variant found at the state, and the true f at the pick in PFN bins relative to the incumbent.
  - Per state and variant: the mean pairwise distance between replicate picks, and the sd of f at the picks.
- *Part B (rollouts).*
  - Full BO runs (n_init = 4, T = 20) of every variant on 1000 tasks of **split 0**: the tuning split, because this chooses π₀.
  - 4 replicates each: the replicate selects the candidate-set nuisance, which the fixed variants ignore.
  - Final regret per (task, variant, replicate): normalized regret (% of f_med − f\*), regret in PFN bins (ECDF units / 0.001), and log regret.
  - f\* is lowered by every run.

**What to look for.**
1. Part B, the decision: the paired difference (replicate mean per task) of `fixed+refine` − `sampled` in normalized regret, relative, with a bootstrap CI over tasks. Also its determinism: the share of tasks with replicate sd of the final regret < 1e-6 bins.
2. Part B, the size of the label noise π₀ induces: the replicate sd of the final normalized regret per task for each variant. `sampled` is the noise the round-1 labels carried.
3. Part B, the tail: the q90 and q95 of the per-task normalized regret, and the failure rate P(final regret > 10 bins), by variant.
4. Part A:
   - the EI ratio (median, q10, share < 0.99) and the replicate pick distance by variant;
   - by step t and by incumbent gap in bins: where does the sampled argmax lose most?

**GO / NO-GO** (rule `analysis/ei_opt.py::ei_opt_verdict`, worthwhile = 5% relative). This decides whether `fixed+refine` becomes π₀ for the next labels.
- **GO**: `fixed+refine` is deterministic (≥ 99% of tasks with replicate sd < 1e-6 bins) and not worse than `sampled` by 5% (the upper CI bound of the relative paired difference < +5%).
- **NO-GO**: not deterministic, or clearly worse (lower bound > +5%).
- **INCONCLUSIVE**: otherwise.

A *better* `fixed+refine` would be a bonus, not required. The point is removing the optimizer noise without losing performance.

**Limits.**
- Noise-free P0, d = 2, T = 20 only (pfn_large valid for n ≤ 100; contexts n ≤ 24).
- Refinement can still miss a better basin of EI that none of the top-8 starts lies in. Part A's EI ratio is relative to the best *found*, not the true maximum.
- GPU non-determinism could make "deterministic" variants differ across replicates; the determinism check measures exactly that.
- 1000 tasks is a pilot size for small effects.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/ei_opt/<sweep_id>"  # set after `just snapshot runs/ei_opt/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.